# Multi‑Step LLM Chain → Knowledge Graph  Notebook

This notebook demonstrates a practical, modular pipeline to build a small **Knowledge Graph (KG)** from unstructured text using a **multi‑step LLM chain**:

1) **Entities** → 2) **Relations** → 3) **Traits (Big Five)** → 4) **Sanity/Dedup** → Graph build

You can run it **offline** via a deterministic `EchoMockLLM`, or switch to a real LLM by plugging your API into the `LLM adapter` cell.

### What you’ll see
- Pydantic **schemas** for entities, relations, trait observations, evidence
- Prompt templates for each stage
- A reusable **Pipeline** class
- A **toy synthetic corpus** (2 short docs)
- Example **queries** on the resulting graph
- **Neo4j export** (CSV + Cypher import script)

**Tip:** Start offline with `EchoMockLLM` to validate the flow. Then switch the `llm` to your provider and re‑run the pipeline.

In [101]:
# Imports
from __future__ import annotations
from typing import List, Dict, Any, Optional, Tuple, Protocol
from dataclasses import dataclass
import json, re, difflib, itertools
from pydantic import BaseModel, Field, ValidationError, constr
import networkx as nx
from pathlib import Path
print("Imports OK")

Imports OK


## LLM Adapter (plug your provider here)

- Implement `.chat(messages, response_format=None) -> str` to return **raw text** from your model.
- Keep temperature low for extraction tasks.
- The notebook defaults to `EchoMockLLM` to stay offline and runnable out‑of‑the‑box.

In [102]:
class LLMClient(Protocol):
    def chat(self, messages: List[Dict[str, str]], response_format: Optional[Dict[str, Any]] = None) -> str:
        ...

class EchoMockLLM:
    """
    Deterministic mock. It detects which stage by scanning the last user message content
    and returns hard-coded JSON for the included toy docs.
    Replace this with your real provider to go live.
    """
    def chat(self, messages, response_format=None):
        q = messages[-1]["content"]
        # ENTITIES stage
        if "Extract ENTITIES" in q or '"entities": [' in q:
            return json.dumps({
                "entities":[
                    {"name":"Dr. Maya Rao","type":"Person","aliases":["Maya Rao","Dr. Rao","Maya"],
                     "mentions":[{"doc_id":"DOC1","sent_id":0,"char_start":4,"char_end":15,"text":"Dr. Maya Rao"}]},
                    {"name":"Aster Labs","type":"Org","aliases":[],
                     "mentions":[{"doc_id":"DOC1","sent_id":0,"char_start":35,"char_end":45,"text":"Aster Labs"}]},
                    {"name":"NovaBio","type":"Org","aliases":[],
                     "mentions":[{"doc_id":"DOC1","sent_id":1,"char_start":33,"char_end":40,"text":"NovaBio"}]},
                    {"name":"Boston","type":"Location","aliases":[],
                     "mentions":[{"doc_id":"DOC1","sent_id":1,"char_start":52,"char_end":58,"text":"Boston"}]},
                    # Second doc entities (DOC2)
                    {"name":"Ben Tan","type":"Person","aliases":["Ben"],
                     "mentions":[{"doc_id":"DOC2","sent_id":0,"char_start":0,"char_end":7,"text":"Ben Tan"}]},
                    {"name":"NovaBio","type":"Org","aliases":[],
                     "mentions":[{"doc_id":"DOC2","sent_id":0,"char_start":24,"char_end":31,"text":"NovaBio"}]},
                    {"name":"Aster Labs","type":"Org","aliases":[],
                     "mentions":[{"doc_id":"DOC2","sent_id":1,"char_start":18,"char_end":28,"text":"Aster Labs"}]}
                ]
            })
        # RELATIONS stage
        if "extract RELATIONS" in q:
            return json.dumps({
                "relations":[
                    {"head":"Dr. Maya Rao","relation":"WORKS_FOR","tail":"Aster Labs","confidence":0.93,
                     "evidence":{"doc_id":"DOC1","sent_id":0,"char_start":0,"char_end":80,
                                 "text":"On 2 May 2024, Dr. Maya Rao joined Aster Labs as Head of Research."}},
                    {"head":"Dr. Maya Rao","relation":"PREVIOUSLY_WORKED_FOR","tail":"NovaBio","confidence":0.88,
                     "evidence":{"doc_id":"DOC1","sent_id":1,"char_start":0,"char_end":80,
                                 "text":"She previously led AI at NovaBio in Boston."}},
                    {"head":"NovaBio","relation":"LOCATED_IN","tail":"Boston","confidence":0.72,
                     "evidence":{"doc_id":"DOC1","sent_id":1,"char_start":35,"char_end":80,
                                 "text":"NovaBio in Boston"}},
                    # DOC2 relations
                    {"head":"Ben Tan","relation":"WORKS_FOR","tail":"NovaBio","confidence":0.86,
                     "evidence":{"doc_id":"DOC2","sent_id":0,"char_start":0,"char_end":70,
                                 "text":"Ben Tan manages clinical trials at NovaBio."}},
                    {"head":"Ben Tan","relation":"COLLABORATES_WITH","tail":"Aster Labs","confidence":0.74,
                     "evidence":{"doc_id":"DOC2","sent_id":1,"char_start":0,"char_end":80,
                                 "text":"He coordinates weekly sprints with Aster Labs"}}
                ]
            })
        # TRAITS stage
        if "Infer personality traits" in q:
            return json.dumps({
                "traits":[
                    {"subject":"Dr. Maya Rao","trait_id":"OCEAN_Conscientiousness","trait_label":"Conscientiousness",
                     "score":0.76,"confidence":0.72,"observed_at":"2024-05-02","method":"LLM-mock",
                     "evidence":{"doc_id":"DOC1","sent_id":0,"char_start":46,"char_end":80,
                                 "text":"joined Aster Labs as Head of Research"}},
                    {"subject":"Ben Tan","trait_id":"OCEAN_Agreeableness","trait_label":"Agreeableness",
                     "score":0.64,"confidence":0.60,"observed_at":"2024-06-03","method":"LLM-mock",
                     "evidence":{"doc_id":"DOC2","sent_id":1,"char_start":0,"char_end":80,
                                 "text":"coordinates weekly sprints with Aster Labs"}}
                ]
            })
        # SANITY stage (we use local merge function; this path unused)
        return "{}"

## Schemas & Prompt Templates
Pydantic models validate JSON returned by the LLM. Prompts are kept concise and task-specific.

In [103]:
class Evidence(BaseModel):
    doc_id: str
    sent_id: int
    char_start: int = Field(..., ge=0)
    char_end: int = Field(..., ge=0)
    text: str

class Entity(BaseModel):
    id: Optional[str] = None
    name: str
    type: constr(strip_whitespace=True)
    aliases: List[str] = Field(default_factory=list)
    mentions: List[Evidence] = Field(default_factory=list)

class Relation(BaseModel):
    head: str
    relation: str
    tail: str
    evidence: Evidence
    confidence: float = Field(0.0, ge=0.0, le=1.0)

class TraitObservation(BaseModel):
    subject: str
    trait_id: str
    trait_label: str
    score: float = Field(..., ge=0.0, le=1.0)
    confidence: float = Field(0.0, ge=0.0, le=1.0)
    observed_at: Optional[str] = None
    method: Optional[str] = None
    evidence: Evidence

SYSTEM_BASE = """You are a careful information extraction assistant.\nAlways output STRICT JSON. Do not add commentary. Use null for missing fields."""

PROMPT_ENTITIES = """
Task: Extract ENTITIES from the text. Types limited to: Person, Org, Project, Location, Date, Product.
Return JSON:
{
  "entities": [
    {
      "name": "...",
      "type": "Person|Org|Project|Location|Date|Product",
      "aliases": ["..."],
      "mentions": [
        {"doc_id":"DOC1","sent_id":0,"char_start":10,"char_end":22,"text":"..."}
      ]
    }
  ]
}
Text:\n```\n{doc_text}\n```
"""

PROMPT_RELATIONS = """
Task: Given the text and the entity list, extract RELATIONS (typed edges).
Use short, conventional relation names in UPPER_SNAKE_CASE, e.g., WORKS_FOR, LOCATED_IN, LEADS, FOUNDED, PARTNERED_WITH, REPORTS_TO.
Return JSON:
{
  "relations": [
    {
      "head":"<entity name as in entities>",
      "relation":"WORKS_FOR",
      "tail":"<entity name as in entities>",
      "confidence":0.0-1.0,
      "evidence":{"doc_id":"DOC1","sent_id":0,"char_start":0,"char_end":10,"text":"..."}
    }
  ]
}
Entities (JSON):
{entities_json}

Text:\n```\n{doc_text}\n```
"""

PROMPT_TRAITS = """
Task: Infer personality traits for PERSON entities ONLY using OCEAN (Big Five):
- Openness, Conscientiousness, Extraversion, Agreeableness, Neuroticism
Return JSON:
{
  "traits":[
    {
      "subject":"<person entity name>",
      "trait_id":"OCEAN_Conscientiousness",
      "trait_label":"Conscientiousness",
      "score":0.0-1.0,
      "confidence":0.0-1.0,
      "observed_at":"YYYY-MM-DD or null",
      "method":"LLM-v1",
      "evidence":{"doc_id":"DOC1","sent_id":0,"char_start":0,"char_end":10,"text":"..."}
    }
  ]
}
Notes:
- Only output traits with textual support; include exactly one evidence span per trait.
- If no clear support, omit that trait for that person.
Entities (JSON):
{entities_json}

Text:\n```\n{doc_text}\n```
"""


## JSON extraction helper & alias merge
The `extract_json` helper unwraps JSON from code fences; `alias_merge` performs quick difflib-based deduplication per entity type.

In [104]:
def extract_json(text: str) -> Dict[str, Any]:
    if not text:
        return {}
    fence = re.findall(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.S)
    candidate = fence[0] if fence else text.strip()
    start = candidate.find("{")
    end = candidate.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("No JSON object found in response.")
    candidate = candidate[start:end+1]
    return json.loads(candidate)

def alias_merge(entities: List[Entity], similarity: float = 0.82) -> List[Entity]:
    by_type: Dict[str, List[Entity]] = {}
    for e in entities:
        by_type.setdefault(e.type, []).append(e)
    merged: List[Entity] = []
    for t, group in by_type.items():
        consumed = set()
        for i, e in enumerate(group):
            if i in consumed:
                continue
            cluster = [e]
            for j in range(i+1, len(group)):
                if j in consumed:
                    continue
                e2 = group[j]
                names1 = [e.name] + e.aliases
                names2 = [e2.name] + e2.aliases
                sim = max(
                    difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio()
                    for a in names1 for b in names2
                )
                if sim >= similarity:
                    cluster.append(e2)
                    consumed.add(j)
            all_names = [c.name for c in cluster]
            canonical = max(all_names, key=len)
            aliases = list(sorted(set(itertools.chain.from_iterable([c.aliases + [c.name] for c in cluster if c.name != canonical]))))
            mentions = list(itertools.chain.from_iterable([c.mentions for c in cluster]))
            merged.append(Entity(id=canonical, name=canonical, type=t, aliases=aliases, mentions=mentions))
    return merged


## Pipeline implementation
Stages: Entities → Relations → Traits → Sanity → Graph build → Example queries

In [105]:
class MultiStepPipeline:
    def __init__(self, llm: LLMClient, doc_id: str = "DOC1"):
        self.llm = llm
        self.doc_id = doc_id

    def extract_entities(self, text: str) -> List[Entity]:
        messages = [
            {"role":"system","content":SYSTEM_BASE},
            {"role":"user","content":PROMPT_ENTITIES.replace("{doc_text}", text)}
        ]
        raw = self.llm.chat(messages)
        data = extract_json(raw)
        ents = []
        for item in data.get("entities", []):
            mentions = []
            for m in item.get("mentions", []):
                if not m.get("doc_id"):
                    m["doc_id"] = self.doc_id
                mentions.append(Evidence(**m))
            ents.append(Entity(name=item["name"], type=item["type"], aliases=item.get("aliases", []), mentions=mentions))
        return ents

    def extract_relations(self, text: str, entities: List[Entity]) -> List[Relation]:
        entities_json = json.dumps([e.dict() for e in entities], ensure_ascii=False, indent=2)
        messages = [
            {"role":"system","content":SYSTEM_BASE},
            {"role":"user","content":PROMPT_RELATIONS.replace("{doc_text}", text).replace("{entities_json}", entities_json)}
        ]
        raw = self.llm.chat(messages)
        data = extract_json(raw)
        rels = []
        for r in data.get("relations", []):
            ev = r["evidence"]
            if not ev.get("doc_id"):
                ev["doc_id"] = self.doc_id
            rels.append(Relation(head=r["head"], relation=r["relation"], tail=r["tail"], confidence=r.get("confidence", 0.0), evidence=Evidence(**ev)))
        return rels

    def extract_traits(self, text: str, entities: List[Entity]) -> List[TraitObservation]:
        entities_json = json.dumps([e.dict() for e in entities], ensure_ascii=False, indent=2)
        messages = [
            {"role":"system","content":SYSTEM_BASE},
            {"role":"user","content":PROMPT_TRAITS.replace("{doc_text}", text).replace("{entities_json}", entities_json)}
        ]
        raw = self.llm.chat(messages)
        data = extract_json(raw)
        out = []
        for t in data.get("traits", []):
            ev = t["evidence"]
            if not ev.get("doc_id"):
                ev["doc_id"] = self.doc_id
            out.append(TraitObservation(subject=t["subject"], trait_id=t["trait_id"], trait_label=t["trait_label"], score=t["score"], confidence=t.get("confidence", 0.0), observed_at=t.get("observed_at"), method=t.get("method"), evidence=Evidence(**ev)))
        return out

    def sanity_merge(self, entities: List[Entity]) -> List[Entity]:
        return alias_merge(entities)

    def canonicalize(self, merged_entities: List[Entity], relations: List[Relation], traits: List[TraitObservation]):
        name_to_id = {e.name: (e.id or e.name) for e in merged_entities}
        for e in merged_entities:
            for a in e.aliases:
                name_to_id.setdefault(a, e.id or e.name)
        def canon(n: str) -> str:
            return name_to_id.get(n, n)
        for r in relations:
            r.head = canon(r.head)
            r.tail = canon(r.tail)
        for t in traits:
            t.subject = canon(t.subject)
        return relations, traits

    def build_graph(self, entities: List[Entity], relations: List[Relation], traits: List[TraitObservation]) -> nx.MultiDiGraph:
        G = nx.MultiDiGraph()
        for e in entities:
            G.add_node(e.id or e.name, label=e.type, aliases=e.aliases, mentions=[m.dict() for m in e.mentions])
        for r in relations:
            G.add_edge(r.head, r.tail, key=f"REL::{r.relation}::{r.evidence.sent_id}", kind="RELATION", relation=r.relation, confidence=r.confidence, evidence=r.evidence.dict())
        for t in traits:
            trait_node = f"TRAIT::{t.trait_id}"
            if not G.has_node(trait_node):
                G.add_node(trait_node, label="Trait", trait_id=t.trait_id, trait_label=t.trait_label)
            G.add_edge(t.subject, trait_node, key=f"TRAIT::{t.trait_id}::{t.evidence.sent_id}", kind="TRAIT", score=t.score, confidence=t.confidence, observed_at=t.observed_at, method=t.method, evidence=t.evidence.dict())
        return G

    def run(self, text: str):
        entities = self.extract_entities(text)
        relations = self.extract_relations(text, entities)
        traits = self.extract_traits(text, entities)
        merged_entities = self.sanity_merge(entities)
        relations, traits = self.canonicalize(merged_entities, relations, traits)
        G = self.build_graph(merged_entities, relations, traits)
        return merged_entities, relations, traits, G

print("Pipeline ready")

Pipeline ready


## Synthetic corpus (toy)
Two short documents with built-in cues for entities, relations, and traits.

In [106]:
DOCS = {
    "DOC1": (
        "On 2 May 2024, Dr. Maya Rao joined Aster Labs as Head of Research. "
        "She previously led AI at NovaBio in Boston."
    ),
    "DOC2": (
        "Ben Tan manages clinical trials at NovaBio. "
        "He coordinates weekly sprints with Aster Labs to ship reliable releases."
    ),
}
print("Loaded", len(DOCS), "docs")

Loaded 2 docs


## Run pipeline on all docs (offline via EchoMockLLM)
You can swap `EchoMockLLM()` with your real provider adapter later.

In [107]:
llm = EchoMockLLM()
pipeline = MultiStepPipeline(llm=llm)

all_entities, all_relations, all_traits = [], [], []
graphs = {}
for doc_id, text in DOCS.items():
    p = MultiStepPipeline(llm=llm, doc_id=doc_id)
    ents, rels, traits, G = p.run(text)
    graphs[doc_id] = G
    all_entities += ents
    all_relations += rels
    all_traits += traits

print(f"Entities: {len(all_entities)} | Relations: {len(all_relations)} | Traits: {len(all_traits)}")

Entities: 10 | Relations: 10 | Traits: 4


## Merge per-doc graphs into one KG (simple overlay)

In [108]:
KG = nx.MultiDiGraph()
for G in graphs.values():
    KG.update(G)
print("Merged KG nodes:", KG.number_of_nodes(), "edges:", KG.number_of_edges())
list(KG.nodes(data=True))[:5]

Merged KG nodes: 7 edges: 14


[('Dr. Maya Rao',
  {'label': 'Person',
   'aliases': [],
   'mentions': [{'doc_id': 'DOC1',
     'sent_id': 0,
     'char_start': 4,
     'char_end': 15,
     'text': 'Dr. Maya Rao'}]}),
 ('Ben Tan',
  {'label': 'Person',
   'aliases': [],
   'mentions': [{'doc_id': 'DOC2',
     'sent_id': 0,
     'char_start': 0,
     'char_end': 7,
     'text': 'Ben Tan'}]}),
 ('Aster Labs',
  {'label': 'Org',
   'aliases': [],
   'mentions': [{'doc_id': 'DOC1',
     'sent_id': 0,
     'char_start': 35,
     'char_end': 45,
     'text': 'Aster Labs'},
    {'doc_id': 'DOC2',
     'sent_id': 1,
     'char_start': 18,
     'char_end': 28,
     'text': 'Aster Labs'}]}),
 ('NovaBio',
  {'label': 'Org',
   'aliases': [],
   'mentions': [{'doc_id': 'DOC1',
     'sent_id': 1,
     'char_start': 33,
     'char_end': 40,
     'text': 'NovaBio'},
    {'doc_id': 'DOC2',
     'sent_id': 0,
     'char_start': 24,
     'char_end': 31,
     'text': 'NovaBio'}]}),
 ('Boston',
  {'label': 'Location',
   'aliases': []

## Example queries (helper functions)

In [109]:
def people_at(G: nx.MultiDiGraph, org_name: str):
    return [
        (u, data.get('relation'), v, data.get('evidence',{}).get('text'))
        for u, v, k, data in G.edges(keys=True, data=True)
        if data.get('kind') == 'RELATION' and data.get('relation') == 'WORKS_FOR' and v == org_name
    ]

def collaborations(G: nx.MultiDiGraph, name: str):
    return [
        (u, v, data.get('evidence',{}).get('text'))
        for u, v, k, data in G.edges(keys=True, data=True)
        if data.get('kind') == 'RELATION' and data.get('relation') == 'COLLABORATES_WITH' and (u == name or v == name)
    ]

def top_trait(G: nx.MultiDiGraph, trait_id: str, k: int = 5):
    scores = {}
    for u, v, key, data in G.edges(keys=True, data=True):
        if data.get('kind') == 'TRAIT' and v == f"TRAIT::{trait_id}":
            scores[u] = max(scores.get(u, 0.0), data.get('score', 0.0))
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]

print("Helpers ready")

Helpers ready


In [110]:
print("People at Aster Labs:")
print(people_at(KG, "Aster Labs"))
print("\nCollaborations with Aster Labs:")
print(collaborations(KG, "Aster Labs"))
print("\nTop Conscientiousness:")
print(top_trait(KG, "OCEAN_Conscientiousness", k=5))

People at Aster Labs:
[('Dr. Maya Rao', 'WORKS_FOR', 'Aster Labs', 'On 2 May 2024, Dr. Maya Rao joined Aster Labs as Head of Research.'), ('Dr. Maya Rao', 'WORKS_FOR', 'Aster Labs', 'On 2 May 2024, Dr. Maya Rao joined Aster Labs as Head of Research.')]

Collaborations with Aster Labs:
[('Ben Tan', 'Aster Labs', 'He coordinates weekly sprints with Aster Labs'), ('Ben Tan', 'Aster Labs', 'He coordinates weekly sprints with Aster Labs')]

Top Conscientiousness:
[('Dr. Maya Rao', 0.76)]


## Neo4j Export (CSV + Cypher)
This section creates simple CSVs for **nodes** and **edges** along with a generated Cypher script to load them into Neo4j.

**Files generated:**
- `kg_nodes.csv` — columns: `id,label,name,aliases`
- `kg_edges.csv` — columns: `start_id,end_id,type,kind,confidence,score,observed_at,relation,method,evidence_json`
- `neo4j_import.cypher` — sample Cypher to create constraints and import the CSVs

In [111]:
import csv, os, json
from pathlib import Path
from string import Template

base = Path("/Users/vivekvardhankolikolla/Downloads")
nodes_path = base / "kg_nodes.csv"
edges_path = base / "kg_edges.csv"
cypher_path = base / "neo4j_import.cypher"

# Nodes CSV
with nodes_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id","label","name","aliases"])  # simple columns
    for n, data in KG.nodes(data=True):
        label = data.get("label", "Thing")
        name = data.get("trait_label") if label == "Trait" else data.get("name", n)
        aliases_val = data.get("aliases", [])
        aliases = ";".join(aliases_val) if isinstance(aliases_val, list) else ""
        w.writerow([n, label, name, aliases])

# Edges CSV
with edges_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([
        "start_id","end_id","type","kind","confidence","score","observed_at",
        "relation","method","evidence_json"
    ])
    for u, v, k, data in KG.edges(keys=True, data=True):
        w.writerow([
            u,
            v,
            data.get("relation") or ("HAS_TRAIT" if data.get("kind") == "TRAIT" else "RELATED_TO"),
            data.get("kind"),
            data.get("confidence",""),
            data.get("score",""),
            data.get("observed_at",""),
            data.get("relation",""),
            data.get("method",""),
            json.dumps(data.get("evidence", {}), ensure_ascii=False)
        ])

# Cypher import script (use Template to avoid brace-escape hell)
cypher_tmpl = Template(r"""
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Thing) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (p:Person) REQUIRE p.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (o:Org) REQUIRE o.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (t:Trait) REQUIRE t.id IS UNIQUE;

LOAD CSV WITH HEADERS FROM 'file:///$nodes' AS row
WITH row,
     CASE row.label WHEN 'Person' THEN 'Person' WHEN 'Org' THEN 'Org' WHEN 'Trait' THEN 'Trait' ELSE 'Thing' END AS lbl
CALL apoc.merge.node([lbl], {id: row.id}, {name: row.name, aliases: row.aliases}) YIELD node
RETURN count(*) as nodes_loaded;

LOAD CSV WITH HEADERS FROM 'file:///$edges' AS row
MATCH (s {id: row.start_id}), (e {id: row.end_id})
CALL apoc.merge.relationship(
  s,
  row.type,
  {},
  {kind: row.kind,
   confidence: toFloat(NULLIF(row.confidence,'')),
   score: toFloat(NULLIF(row.score,'')),
   observed_at: NULLIF(row.observed_at,''),
   method: NULLIF(row.method,''),
   evidence_json: row.evidence_json},
  e
) YIELD rel
RETURN count(*) as rels_loaded;
""")

cypher = cypher_tmpl.substitute(nodes=nodes_path.name, edges=edges_path.name)
cypher_path.write_text(cypher, encoding="utf-8")

print("Wrote:")
print(" -", nodes_path)
print(" -", edges_path)
print(" -", cypher_path)


Wrote:
 - /Users/vivekvardhankolikolla/Downloads/kg_nodes.csv
 - /Users/vivekvardhankolikolla/Downloads/kg_edges.csv
 - /Users/vivekvardhankolikolla/Downloads/neo4j_import.cypher


In [112]:
pip install openai

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pip/__main__.py", line 8, in <module>
    if sys.path[0] in ("", os.getcwd()):
                           ^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory
Note: you may need to restart the kernel to use updated packages.


In [113]:
## Switching to a real LLM
## Replace `EchoMockLLM()` with your provider adapter. Example (OpenAI):


from openai import OpenAI
class OpenAILLM:
    def __init__(self, model="gpt-4o-mini", api_key=None):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def chat(self, messages, response_format=None):
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.1,
            response_format={"type":"json_object"}
        )
        return resp.choices[0].message.content


llm = OpenAILLM(model="gpt-4o-mini", api_key="")
pipeline = MultiStepPipeline(llm=llm, doc_id="DOCX")


## Switching to a real LLM
## Replace `EchoMockLLM()` with your provider adapter. Example (OpenAI):


from openai import OpenAI
class OpenAILLM:
    def __init__(self, model="gpt-4o-mini", api_key=None):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def chat(self, messages, response_format=None):
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.1,
            response_format={"type":"json_object"}
        )
        return resp.choices[0].message.content


llm = OpenAILLM(model="gpt-4o-mini", api_key="")
pipeline = MultiStepPipeline(llm=llm, doc_id="DOCX")
